# 03 - Huan Luyen va Danh Gia Mo Hinh

Notebook nay thuc hien:
- Trich xuat dac trung
- Huan luyen SVM va KNN
- Danh gia va hien thi ket qua

In [ ]:
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sys.path.insert(0, os.path.join('..', 'src'))
from preprocessing import load_dataset, convert_to_hsv, quantize_colors_hsv
from color_correlogram import auto_correlogram_fast
from color_histogram import color_histogram

## 1. Tai du lieu va trich xuat dac trung

In [ ]:
# Kiem tra xem da co file features chua
FEATURES_DIR = os.path.join('..', 'data', 'features')

if os.path.exists(os.path.join(FEATURES_DIR, 'correlogram_hsv.npy')):
    print('Da co file features, tai truc tiep...')
    X_corr = np.load(os.path.join(FEATURES_DIR, 'correlogram_hsv.npy'))
    X_hist = np.load(os.path.join(FEATURES_DIR, 'histogram_hsv.npy'))
    y = np.load(os.path.join(FEATURES_DIR, 'labels.npy'))
    class_names = np.load(os.path.join(FEATURES_DIR, 'class_names.npy'), allow_pickle=True)
else:
    print('Chua co file features, trich xuat tu dau...')
    DATA_DIR = os.path.join('..', 'data', 'corel-1k')
    images, labels, paths = load_dataset(DATA_DIR)
    
    le = LabelEncoder()
    y = le.fit_transform(labels)
    class_names = le.classes_
    
    import cv2
    X_corr = []
    X_hist = []
    for i, img in enumerate(images):
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        q, nc = quantize_colors_hsv(hsv)
        X_corr.append(auto_correlogram_fast(q, nc))
        X_hist.append(color_histogram(q, nc))
        if (i+1) % 100 == 0:
            print(f'  {i+1}/{len(images)}')
    
    X_corr = np.array(X_corr)
    X_hist = np.array(X_hist)

print(f'Correlogram features: {X_corr.shape}')
print(f'Histogram features:   {X_hist.shape}')
print(f'Labels: {y.shape}')
print(f'Classes: {list(class_names)}')

## 2. Huan luyen va danh gia cac mo hinh

In [ ]:
# Dinh nghia cac mo hinh
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'SVM': Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='rbf', C=10, gamma='scale'))]),
    'KNN': Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=5, weights='distance'))]),
}

results = {}

# Huan luyen voi Correlogram
print('=== Huan luyen voi Color Correlogram (HSV) ===')
for name, model in models.items():
    start = time.time()
    scores = cross_val_score(model, X_corr, y, cv=cv, scoring='accuracy')
    elapsed = time.time() - start
    results[f'Corr+{name}'] = scores
    print(f'  {name}: {scores.mean():.4f} (+/- {scores.std():.4f}) [{elapsed:.1f}s]')

# Huan luyen voi Histogram (baseline)
print('\n=== Huan luyen voi Color Histogram (HSV) - BASELINE ===')
for name, model in models.items():
    start = time.time()
    scores = cross_val_score(model, X_hist, y, cv=cv, scoring='accuracy')
    elapsed = time.time() - start
    results[f'Hist+{name}'] = scores
    print(f'  {name}: {scores.mean():.4f} (+/- {scores.std():.4f}) [{elapsed:.1f}s]')

In [ ]:
# Ve bieu do so sanh
fig, ax = plt.subplots(figsize=(12, 6))

names = list(results.keys())
means = [results[n].mean() * 100 for n in names]
stds = [results[n].std() * 100 for n in names]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#E91E63', '#9C27B0']

bars = ax.bar(range(len(names)), means, yerr=stds, color=colors[:len(names)],
              edgecolor='black', capsize=5, width=0.6)

for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{m:.1f}%', ha='center', fontweight='bold', fontsize=11)

ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=10, rotation=20, ha='right')
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('So sanh Accuracy: Correlogram vs Histogram x SVM/KNN/RF',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'notebook_accuracy_comparison.png'), dpi=150)
plt.show()

## 3. Confusion Matrix cho mo hinh tot nhat

In [ ]:
# Chay cross-val predict cho SVM + Correlogram
best_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(kernel='rbf', C=10, gamma='scale'))
])

y_pred = cross_val_predict(best_model, X_corr, y, cv=cv)

# Confusion matrix
cm = confusion_matrix(y, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Du doan (Predicted)', fontsize=12)
ax.set_ylabel('Thuc te (Actual)', fontsize=12)
ax.set_title('Confusion Matrix - Correlogram (HSV) + SVM', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'notebook_confusion_matrix.png'), dpi=150)
plt.show()

# Classification report
print('\nClassification Report:')
print(classification_report(y, y_pred, target_names=class_names))

## 4. Phan tich ket qua

In [ ]:
# Accuracy theo tung lop
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(class_names, per_class_acc, color='coral', edgecolor='black')

for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{acc:.0f}%', ha='center', fontweight='bold')

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Accuracy theo tung lop - Correlogram + SVM', fontsize=14, fontweight='bold')
ax.set_ylim(0, 110)
plt.xticks(rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nLop chinh xac nhat: {class_names[np.argmax(per_class_acc)]} ({per_class_acc.max():.0f}%)')
print(f'Lop kem nhat:       {class_names[np.argmin(per_class_acc)]} ({per_class_acc.min():.0f}%)')